> **Weather-aware workflow note:** Linear Regression, Random Forest, and XGBoost train with current-time weather measurements plus time and historical-power features. Persistence remains the weather-free benchmark.

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
plant1 = pd.read_csv("../data/final/plant1_features.csv")
plant2 = pd.read_csv("../data/final/plant2_features.csv")

plant1["DATE_TIME"] = pd.to_datetime(plant1["DATE_TIME"])
plant2["DATE_TIME"] = pd.to_datetime(plant2["DATE_TIME"])

print("Plant 1 shape:", plant1.shape)
print("Plant 2 shape:", plant2.shape)

Plant 1 shape: (66337, 27)
Plant 2 shape: (65466, 27)


In [3]:
feature_columns = [
    "hour",
    "minute",
    "day",
    "day_of_week",
    "day_of_year",
    "month",
    "hour_sin",
    "hour_cos",
    "IRRADIATION",
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
    "AC_POWER_lag_1",
    "AC_POWER_lag_4",
    "AC_POWER_lag_8",
    "AC_POWER_lag_96",
    "AC_POWER_rolling_mean_4",
    "AC_POWER_rolling_mean_8"
]

target_column = "TARGET_AC_POWER"

print("Number of features:", len(feature_columns))
print("Target:", target_column)

Number of features: 17
Target: TARGET_AC_POWER


In [4]:
def chronological_split(df, train_ratio=0.70, val_ratio=0.15):
    """Split chronologically and remove rows whose target crosses a cutoff."""
    df = df.sort_values("DATE_TIME").copy()
    unique_times = np.sort(df["DATE_TIME"].unique())
    n_times = len(unique_times)
    train_end = int(n_times * train_ratio)
    val_end = int(n_times * (train_ratio + val_ratio))
    train_end_time = pd.Timestamp(unique_times[train_end])
    val_end_time = pd.Timestamp(unique_times[val_end])
    target_time = df["DATE_TIME"] + pd.Timedelta(minutes=15)

    train = df[(df["DATE_TIME"] < train_end_time) & (target_time < train_end_time)].copy()
    validation = df[
        (df["DATE_TIME"] >= train_end_time)
        & (df["DATE_TIME"] < val_end_time)
        & (target_time < val_end_time)
    ].copy()
    test = df[df["DATE_TIME"] >= val_end_time].copy()
    return train, validation, test

In [5]:
plant1_train, plant1_val, plant1_test = chronological_split(plant1)

plant2_train, plant2_val, plant2_test = chronological_split(plant2)

print("Plant 1:")
print("Train:", plant1_train.shape)
print("Validation:", plant1_val.shape)
print("Test:", plant1_test.shape)

print("\nPlant 2:")
print("Train:", plant2_train.shape)
print("Validation:", plant2_val.shape)
print("Test:", plant2_test.shape)

Plant 1:
Train: (46163, 27)
Validation: (10054, 27)
Test: (10076, 27)

Plant 2:
Train: (44588, 27)
Validation: (10406, 27)
Test: (10428, 27)


In [6]:
X1_train = plant1_train[feature_columns]
y1_train = plant1_train[target_column]

X1_val = plant1_val[feature_columns]
y1_val = plant1_val[target_column]

X1_test = plant1_test[feature_columns]
y1_test = plant1_test[target_column]


X2_train = plant2_train[feature_columns]
y2_train = plant2_train[target_column]

X2_val = plant2_val[feature_columns]
y2_val = plant2_val[target_column]

X2_test = plant2_test[feature_columns]
y2_test = plant2_test[target_column]

In [7]:
print("Plant 1:")
print("X_train:", X1_train.shape)
print("y_train:", y1_train.shape)
print("X_val:", X1_val.shape)
print("X_test:", X1_test.shape)

print("\nPlant 2:")
print("X_train:", X2_train.shape)
print("y_train:", y2_train.shape)
print("X_val:", X2_val.shape)
print("X_test:", X2_test.shape)

Plant 1:
X_train: (46163, 17)
y_train: (46163,)
X_val: (10054, 17)
X_test: (10076, 17)

Plant 2:
X_train: (44588, 17)
y_train: (44588,)
X_val: (10406, 17)
X_test: (10428, 17)


In [8]:
# What this cell does:
# Train a Linear Regression model using only the training data.
# The model learns the relationship between our 17 engineered features
# and the 15-minute-ahead target: TARGET_AC_POWER.

linear_model_plant1 = LinearRegression()
linear_model_plant2 = LinearRegression()

linear_model_plant1.fit(X1_train, y1_train)
linear_model_plant2.fit(X2_train, y2_train)

print("Linear Regression trained successfully.")

Linear Regression trained successfully.


In [9]:
# What this cell does:
# Generate predictions for the validation data using
# the trained Linear Regression models.
#
# The models were trained on the training data only.
# Now we use them to predict the unseen validation period.

y1_val_pred = linear_model_plant1.predict(X1_val)
y2_val_pred = linear_model_plant2.predict(X2_val)

print("Plant 1 validation predictions:", y1_val_pred.shape)
print("Plant 2 validation predictions:", y2_val_pred.shape)

Plant 1 validation predictions: (10054,)
Plant 2 validation predictions: (10406,)


In [10]:
# What this cell does:
# Evaluate Linear Regression on the validation data using:
# MAE  -> average absolute prediction error
# RMSE -> gives more penalty to large errors
# R²   -> proportion of target variation explained by the model

lr_plant1_val_mae = mean_absolute_error(y1_val, y1_val_pred)
lr_plant1_val_rmse = np.sqrt(mean_squared_error(y1_val, y1_val_pred))
lr_plant1_val_r2 = r2_score(y1_val, y1_val_pred)

lr_plant2_val_mae = mean_absolute_error(y2_val, y2_val_pred)
lr_plant2_val_rmse = np.sqrt(mean_squared_error(y2_val, y2_val_pred))
lr_plant2_val_r2 = r2_score(y2_val, y2_val_pred)

print("Plant 1 - Linear Regression Validation")
print("MAE :", lr_plant1_val_mae)
print("RMSE:", lr_plant1_val_rmse)
print("R²  :", lr_plant1_val_r2)

print("\nPlant 2 - Linear Regression Validation")
print("MAE :", lr_plant2_val_mae)
print("RMSE:", lr_plant2_val_rmse)
print("R²  :", lr_plant2_val_r2)

Plant 1 - Linear Regression Validation
MAE : 59.43736508367133
RMSE: 92.54896568078921
R²  : 0.9366533772407428

Plant 2 - Linear Regression Validation
MAE : 78.67210418330679
RMSE: 152.45061521208032
R²  : 0.7699003893808483


In [11]:
# What this cell does:
# Create Persistence predictions on the validation period.
# Persistence predicts the next 15-minute AC power using
# the current AC_POWER value.

y1_val_persistence = plant1_val["AC_POWER"]
y2_val_persistence = plant2_val["AC_POWER"]

print("Plant 1 persistence validation predictions:", y1_val_persistence.shape)
print("Plant 2 persistence validation predictions:", y2_val_persistence.shape)

Plant 1 persistence validation predictions: (10054,)
Plant 2 persistence validation predictions: (10406,)


In [12]:
# What this cell does:
# Calculate MAE, RMSE and R² for the Persistence baseline
# on exactly the same validation period used by Linear Regression.

persistence_plant1_val_mae = mean_absolute_error(
    y1_val, y1_val_persistence
)

persistence_plant1_val_rmse = np.sqrt(
    mean_squared_error(y1_val, y1_val_persistence)
)

persistence_plant1_val_r2 = r2_score(
    y1_val, y1_val_persistence
)


persistence_plant2_val_mae = mean_absolute_error(
    y2_val, y2_val_persistence
)

persistence_plant2_val_rmse = np.sqrt(
    mean_squared_error(y2_val, y2_val_persistence)
)

persistence_plant2_val_r2 = r2_score(
    y2_val, y2_val_persistence
)


print("Plant 1 - Persistence Validation")
print("MAE :", persistence_plant1_val_mae)
print("RMSE:", persistence_plant1_val_rmse)
print("R²  :", persistence_plant1_val_r2)

print("\nPlant 2 - Persistence Validation")
print("MAE :", persistence_plant2_val_mae)
print("RMSE:", persistence_plant2_val_rmse)
print("R²  :", persistence_plant2_val_r2)

Plant 1 - Persistence Validation
MAE : 51.941933381259204
RMSE: 100.82162916785327
R²  : 0.9248225225893167

Plant 2 - Persistence Validation
MAE : 55.59200768928042
RMSE: 122.04423857837519
R²  : 0.8525339370562894


In [13]:
# What this cell does:
# Import the Random Forest regression algorithm.
# Random Forest combines multiple decision trees to learn
# nonlinear relationships between features and AC power.

from sklearn.ensemble import RandomForestRegressor

In [14]:
# What this cell does:
# Train separate Random Forest regression models for Plant 1 and Plant 2.
#
# n_estimators = 200:
# Build 200 decision trees.
#
# random_state = 42:
# Makes the experiment reproducible.
#
# n_jobs = -1:
# Use all available CPU cores.

rf_model_plant1 = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model_plant2 = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model_plant1.fit(X1_train, y1_train)
rf_model_plant2.fit(X2_train, y2_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [15]:
# What this cell does:
# Use the trained Random Forest models to predict
# the unseen validation period.

y1_rf_val_pred = rf_model_plant1.predict(X1_val)
y2_rf_val_pred = rf_model_plant2.predict(X2_val)

print("Plant 1 validation predictions:", y1_rf_val_pred.shape)
print("Plant 2 validation predictions:", y2_rf_val_pred.shape)

Plant 1 validation predictions: (10054,)
Plant 2 validation predictions: (10406,)


In [16]:
# What this cell does:
# Evaluate Random Forest using MAE, RMSE and R²
# on the validation period.

rf_plant1_val_mae = mean_absolute_error(
    y1_val, y1_rf_val_pred
)

rf_plant1_val_rmse = np.sqrt(
    mean_squared_error(y1_val, y1_rf_val_pred)
)

rf_plant1_val_r2 = r2_score(
    y1_val, y1_rf_val_pred
)


rf_plant2_val_mae = mean_absolute_error(
    y2_val, y2_rf_val_pred
)

rf_plant2_val_rmse = np.sqrt(
    mean_squared_error(y2_val, y2_rf_val_pred)
)

rf_plant2_val_r2 = r2_score(
    y2_val, y2_rf_val_pred
)


print("Plant 1 - Random Forest Validation")
print("MAE :", rf_plant1_val_mae)
print("RMSE:", rf_plant1_val_rmse)
print("R²  :", rf_plant1_val_r2)

print("\nPlant 2 - Random Forest Validation")
print("MAE :", rf_plant2_val_mae)
print("RMSE:", rf_plant2_val_rmse)
print("R²  :", rf_plant2_val_r2)

Plant 1 - Random Forest Validation
MAE : 56.662081633823064
RMSE: 108.7273239216848
R²  : 0.9125705542940742

Plant 2 - Random Forest Validation
MAE : 76.97041492740465
RMSE: 168.00346570525556
R²  : 0.7205565060727475


In [17]:
# What this cell does:
# Import XGBRegressor, the gradient-boosting regression algorithm
# we will use as our main classical ML forecasting model.

from xgboost import XGBRegressor

print("XGBoost imported successfully.")

XGBoost imported successfully.


In [18]:
# What this cell does:
# Create separate XGBoost regression models for Plant 1 and Plant 2.
#
# n_estimators = number of boosting trees
# learning_rate = contribution of each tree
# max_depth = maximum depth of each tree
# random_state = reproducible results
#
# These are initial parameters, not final hyperparameter tuning.

xgb_model_plant1 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    objective="reg:squarederror",
    n_jobs=-1
)

xgb_model_plant2 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    objective="reg:squarederror",
    n_jobs=-1
)

print("XGBoost models created successfully.")

XGBoost models created successfully.


In [19]:
# What this cell does:
# Train XGBoost using only the training data.
#
# The validation data remains unseen during training.

xgb_model_plant1.fit(X1_train, y1_train)
xgb_model_plant2.fit(X2_train, y2_train)

print("XGBoost trained successfully.")

XGBoost trained successfully.


In [20]:
# What this cell does:
# Generate predictions for the unseen validation period
# using the trained XGBoost models.

y1_xgb_val_pred = xgb_model_plant1.predict(X1_val)
y2_xgb_val_pred = xgb_model_plant2.predict(X2_val)

print("Plant 1 validation predictions:", y1_xgb_val_pred.shape)
print("Plant 2 validation predictions:", y2_xgb_val_pred.shape)

Plant 1 validation predictions: (10054,)
Plant 2 validation predictions: (10406,)


In [21]:
# What this cell does:
# Calculate MAE, RMSE and R² for XGBoost
# on the validation period.

xgb_plant1_val_mae = mean_absolute_error(
    y1_val, y1_xgb_val_pred
)

xgb_plant1_val_rmse = np.sqrt(
    mean_squared_error(y1_val, y1_xgb_val_pred)
)

xgb_plant1_val_r2 = r2_score(
    y1_val, y1_xgb_val_pred
)


xgb_plant2_val_mae = mean_absolute_error(
    y2_val, y2_xgb_val_pred
)

xgb_plant2_val_rmse = np.sqrt(
    mean_squared_error(y2_val, y2_xgb_val_pred)
)

xgb_plant2_val_r2 = r2_score(
    y2_val, y2_xgb_val_pred
)


print("Plant 1 - XGBoost Validation")
print("MAE :", xgb_plant1_val_mae)
print("RMSE:", xgb_plant1_val_rmse)
print("R²  :", xgb_plant1_val_r2)

print("\nPlant 2 - XGBoost Validation")
print("MAE :", xgb_plant2_val_mae)
print("RMSE:", xgb_plant2_val_rmse)
print("R²  :", xgb_plant2_val_r2)

Plant 1 - XGBoost Validation
MAE : 50.68129248902174
RMSE: 100.28584484564865
R²  : 0.9256194128876528

Plant 2 - XGBoost Validation
MAE : 75.12665406036996
RMSE: 166.13263851927783
R²  : 0.7267454222145819


In [22]:
# What this cell does:
# Combine the validation results of all four forecasting approaches
# into one table so we can compare them using the same dataset split
# and the same evaluation metrics.

validation_results = pd.DataFrame({
    "Plant": [
        "Plant 1", "Plant 1", "Plant 1", "Plant 1",
        "Plant 2", "Plant 2", "Plant 2", "Plant 2"
    ],

    "Model": [
        "Persistence",
        "Linear Regression",
        "Random Forest",
        "XGBoost",
        "Persistence",
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],

    "MAE": [
        persistence_plant1_val_mae,
        lr_plant1_val_mae,
        rf_plant1_val_mae,
        xgb_plant1_val_mae,
        persistence_plant2_val_mae,
        lr_plant2_val_mae,
        rf_plant2_val_mae,
        xgb_plant2_val_mae
    ],

    "RMSE": [
        persistence_plant1_val_rmse,
        lr_plant1_val_rmse,
        rf_plant1_val_rmse,
        xgb_plant1_val_rmse,
        persistence_plant2_val_rmse,
        lr_plant2_val_rmse,
        rf_plant2_val_rmse,
        xgb_plant2_val_rmse
    ],

    "R2": [
        persistence_plant1_val_r2,
        lr_plant1_val_r2,
        rf_plant1_val_r2,
        xgb_plant1_val_r2,
        persistence_plant2_val_r2,
        lr_plant2_val_r2,
        rf_plant2_val_r2,
        xgb_plant2_val_r2
    ]
})

validation_results

,Plant,Model,MAE,RMSE,R2
0,Plant 1,Persistence,51.941933,100.821629,0.924823
1,Plant 1,Linear Regression,59.437365,92.548966,0.936653
2,Plant 1,Random Forest,56.662082,108.727324,0.912571
3,Plant 1,XGBoost,50.681292,100.285845,0.925619
4,Plant 2,Persistence,55.592008,122.044239,0.852534
5,Plant 2,Linear Regression,78.672104,152.450615,0.769900
6,Plant 2,Random Forest,76.970415,168.003466,0.720557
7,Plant 2,XGBoost,75.126654,166.132639,0.726745


In [23]:
# What this cell does:
# Save the complete validation model comparison to the
# outputs/metrics/ folder for later analysis.

import os

METRICS_PATH = "../outputs/metrics/"
os.makedirs(METRICS_PATH, exist_ok=True)

validation_results.to_csv(
    METRICS_PATH + "validation_model_results.csv",
    index=False
)

print("Validation model comparison saved successfully.")

Validation model comparison saved successfully.


In [24]:
# What this cell does:
# Save the trained Linear Regression, Random Forest,
# and XGBoost models so they can be reused later
# without retraining.

import joblib
import os

MODEL_PATH = "../models/"

os.makedirs(MODEL_PATH + "linear_regression/", exist_ok=True)
os.makedirs(MODEL_PATH + "random_forest/", exist_ok=True)
os.makedirs(MODEL_PATH + "xgboost/", exist_ok=True)

# Linear Regression
joblib.dump(
    linear_model_plant1,
    MODEL_PATH + "linear_regression/plant1_model.pkl"
)

joblib.dump(
    linear_model_plant2,
    MODEL_PATH + "linear_regression/plant2_model.pkl"
)

# Random Forest
joblib.dump(
    rf_model_plant1,
    MODEL_PATH + "random_forest/plant1_model.pkl"
)

joblib.dump(
    rf_model_plant2,
    MODEL_PATH + "random_forest/plant2_model.pkl"
)

# XGBoost
joblib.dump(
    xgb_model_plant1,
    MODEL_PATH + "xgboost/plant1_model.pkl"
)

joblib.dump(
    xgb_model_plant2,
    MODEL_PATH + "xgboost/plant2_model.pkl"
)

print("All trained models saved successfully.")

All trained models saved successfully.
